# 2. Classifying cards: oracle text into effect verbs

This is the layer that decides what the engine *could* know about a card.

The design rule from `ARCHITECTURE.md`: **cards are never modelled as fixed
fields** like "a creature with power and toughness". They are data plus a tuple of
`Effect(verb, params)`. Growing the engine means implementing more verbs, not
rewriting the model — which is the only reason "eventually the entire game" is
reachable rather than a second project.

In [1]:
from pathlib import Path

import duckdb

REPO = Path.cwd()
while not (REPO / "data" / "cards.duckdb").exists() and REPO != REPO.parent:
    REPO = REPO.parent

DB = REPO / "data" / "cards.duckdb"
con = duckdb.connect(str(DB), read_only=True)


def q(sql, params=None, limit=12):
    # Run a query and print it as a small table.
    rows = con.execute(sql, params or []).fetchall()
    names = [d[0] for d in con.description]
    shown = rows[:limit]
    if not shown:
        print("  ".join(map(str, names)))
        print("(no rows)")
        return rows
    widths = [max([len(str(n))] + [len(str(r[i])[:38]) for r in shown])
              for i, n in enumerate(names)]
    print("  ".join(str(n).ljust(w) for n, w in zip(names, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v)[:38].ljust(w) for v, w in zip(row, widths)))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows")
    return rows

In [2]:
from manalab.cards.store import CardStore

store = CardStore(DB)

for name in ["Lightning Bolt", "Murder", "Divination", "Giant Growth",
             "Vampire Nighthawk", "Craterhoof Behemoth"]:
    card = store.by_name(name)
    print(f"{name}")
    print(f"    text: {card.oracle_text!r}")
    for effect in card.effects:
        print(f"      -> {effect.verb:<16} {dict(effect.params)}")
    print()

Lightning Bolt
    text: 'Lightning Bolt deals 3 damage to any target.'
      -> deal_damage      {'amount': 3, 'target': 'any target'}

Murder
    text: 'Destroy target creature.'
      -> destroy          {'target': 'target creature'}

Divination
    text: 'Draw two cards.'
      -> draw             {'amount': 2}

Giant Growth
    text: 'Target creature gets +3/+3 until end of turn.'
      -> pump             {'target': 'Target creature', 'amount': '+3/+3'}

Vampire Nighthawk
    text: 'Flying\nDeathtouch (Any amount of damage this deals to a creature is enough to destroy it.)\nLifelink (Damage dealt by this creature also causes you to gain that much life.)'

Craterhoof Behemoth
    text: 'Haste\nWhen this creature enters, creatures you control gain trample and get +X/+X until end of turn, where X is the number of creatures you control.'
      -> pump             {'target': 'creatures you control gain trample and', 'amount': '+X/+X'}
      -> grant_ability    {'ability': 'trample and

## Nothing is skipped

Every clause is either matched to a verb or recorded as unparsed. A parser that
silently drops what it cannot read reports excellent coverage and is useless,
because you cannot tell which cards it lied about.

In [3]:
import collections

from manalab.cards import effects as E

verbs = collections.Counter()
cards_scanned = 0
for card in store.all_cards():
    if len(card.color_identity) != 1 or card.has_faces:
        continue
    cards_scanned += 1
    for effect in card.effects:
        verbs[effect.verb] += 1

print(f"scanned {cards_scanned:,} single-colour cards\n")
print("most common verbs:")
for verb, n in verbs.most_common(20):
    print(f"  {n:>6,}  {verb}")

scanned 24,773 single-colour cards

most common verbs:
   4,378  keyword
   3,847  pump
   2,262  draw
   2,122  create_token
   1,945  put_counters
   1,860  grant_ability
   1,794  exile
   1,710  deal_damage
   1,451  destroy
   1,448  sacrifice
   1,401  ability_word
     993  gain_life
     913  shuffle
     899  reveal
     838  discard
     831  return_from_graveyard
     796  untap
     731  grant_text
     716  tap
     696  lose_life


## What the engine actually executes

The parser knows about forty-odd verbs. The *engine* implements five of them, plus
a set of combat keywords. That gap is the roadmap, and `engine.fidelity` is where
the two lists are reconciled — deliberately in one place, because a second copy
would drift.

In [4]:
from manalab.engine import fidelity, resolve

print("verbs the engine executes on resolution:")
print("   ", ", ".join(sorted(resolve.IMPLEMENTED)))
print("\ncombat keywords the engine implements:")
print("   ", ", ".join(sorted(fidelity.COMBAT_KEYWORDS)))
print(f"\nverbs the parser can produce: {len(verbs)}")
print(f"verbs the engine executes:    {len(resolve.IMPLEMENTED)}")

verbs the engine executes on resolution:
    deal_damage, destroy, draw, gain_life, pump

combat keywords the engine implements:
    deathtouch, defender, double strike, first strike, flying, haste, lifelink, menace, reach, trample, vigilance

verbs the parser can produce: 67
verbs the engine executes:    5


## The target language

Verbs are only half of it. `"deal_damage"` needs to know *at what*, and targets
arrive as English. `engine.targets.parse` recognises four phrases exactly and
**refuses the other 353**.

Refusing is the whole safety property. Guessing that `"target nonblack creature"`
means "any creature" would make illegal plays legal, and the resulting win rates
would look completely reasonable.

In [5]:
from manalab.engine import targets as T

for phrase in ["any target", "target creature", "target creature you control",
               "target player", "target nonblack creature", "all creatures",
               "target attacking or blocking creature",
               "target creature an opponent controls"]:
    spec = T.parse(phrase)
    print(f"  {phrase:<42} {spec.kind.value if spec else 'REFUSED'}")

  any target                                 any
  target creature                            creature
  target creature you control                creature_you_control
  target player                              player
  target nonblack creature                   REFUSED
  all creatures                              REFUSED
  target attacking or blocking creature      REFUSED
  target creature an opponent controls       REFUSED


In [6]:
# The distribution the four recognised phrases were chosen from.
TIER1 = resolve.IMPLEMENTED
phrases = collections.Counter()
for card in store.all_cards():
    if len(card.color_identity) != 1 or card.has_faces or card.is_permanent:
        continue
    if not card.effects or not ({e.verb for e in card.effects} <= TIER1):
        continue
    for effect in card.effects:
        phrases[str(effect.get("target", "<no target>")).lower()] += 1

print("target phrases among spells whose verbs the engine implements:\n")
for phrase, n in phrases.most_common(10):
    mark = "OK " if T.parse(None if phrase == "<no target>" else phrase) else "no "
    print(f"  {mark} {n:>5}  {phrase}")
print(f"\n{len(phrases)} distinct phrases in total — a very long tail.")

target phrases among spells whose verbs the engine implements:

  OK    376  <no target>
  OK    220  target creature
  OK     61  any target
  no     34  creatures you control
  no     27  all creatures
  no     23  target creature or planeswalker
  OK     22  target creature you control
  no     22  target attacking or blocking creature
  no     20  target artifact or enchantment
  no     20  target land

355 distinct phrases in total — a very long tail.


## Coverage, reported honestly

Two numbers that are easy to conflate and mean different things:

- **share of all cards** — includes cards whose text was never read.
- **share of readable cards** — the parser's actual hit rate.

They were once the same number, which flattered the parser by counting 886
face-carrying cards as successes it had never seen a word of.

In [7]:
total = readable = fully_parsed = vanilla = unread = 0
for card in store.all_cards():
    total += 1
    if card.has_faces and not card.oracle_text:
        unread += 1
        continue
    if not card.oracle_text:
        vanilla += 1
        continue
    readable += 1
    if card.effects:
        fully_parsed += 1

print(f"  cards in the corpus     {total:>7,}")
print(f"  genuinely vanilla       {vanilla:>7,}")
print(f"  text never read (faces) {unread:>7,}")
print(f"  readable text           {readable:>7,}")
print(f"  ...producing verbs      {fully_parsed:>7,}   "
      f"({100 * fully_parsed / readable:.1f}% of readable)")

  cards in the corpus      35,142
  genuinely vanilla           350
  text never read (faces)     886
  readable text            33,906
  ...producing verbs       29,875   (88.1% of readable)


## The gap list is the roadmap

`Pool.gaps()` ranks *why* cards were set aside. It is the only sensible order to
implement in: work the top of the list, re-measure, repeat.

In [8]:
from manalab.evaluate import pool as P

for code_letter, name in {"W": "white", "U": "blue", "B": "black",
                          "R": "red", "G": "green"}.items():
    pool = P.mono_color(store, code_letter, fmt="modern")
    print(f"{name}  ({pool.size} usable, {len(pool.text_tier)} set aside)")
    for reason, n in pool.gaps()[:3]:
        print(f"      {n:>5}  {reason}")
    print()

white  (187 usable, 3136 set aside)
       2325  rules text the engine does not execute
        945  unimplemented keywords
        842  verb not implemented



blue  (122 usable, 3123 set aside)
       2093  rules text the engine does not execute
       1319  verb not implemented
       1040  unimplemented keywords



black  (137 usable, 3166 set aside)
       2147  rules text the engine does not execute
       1271  verb not implemented
        940  unimplemented keywords



red  (151 usable, 3121 set aside)
       2089  rules text the engine does not execute
       1055  verb not implemented
        794  unimplemented keywords



green  (187 usable, 3009 set aside)
       2174  rules text the engine does not execute
       1047  verb not implemented
       1018  unimplemented keywords



## What to take into notebook 3

- Cards are data plus `Effect(verb, params)`; nothing is silently dropped.
- The parser produces far more verbs than the engine executes — that gap is
  explicit and lives in one place.
- Targets are a second, separate vocabulary, and unknown phrases are refused.
- Coverage is reported as share-of-readable *and* share-of-all, because
  conflating them hides unread cards.

Next: getting cards back out — queries, pools, and the fidelity split.